# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [3]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../data/co2_emissions.csv')

# Filter to Asia only
asia = df[df['Region'] == 'Asia']

highlight = 'China'  # country to highlight — change as you like

# Build color map
color_map = {c: '#2E75B6' if c == highlight else '#DDDDDD'
             for c in asia['Country'].unique()}

fig = go.Figure()

# Add one trace per country
for country in asia['Country'].unique():
    d = asia[asia['Country'] == country].sort_values('Year')
    is_highlight = country == highlight

    fig.add_trace(go.Scatter(
        x=d['Year'],
        y=d['CO2_Mt'],
        mode='lines',
        name=country,
        line=dict(
            color=color_map[country],
            width=3 if is_highlight else 1.2
        ),
        showlegend=False
    ))

# Direct label at the end of the highlighted line
last = asia[(asia['Country'] == highlight) & (asia['Year'] == asia['Year'].max())]

fig.add_annotation(
    x=last['Year'].values[0],
    y=last['CO2_Mt'].values[0],
    text=f'<b>{highlight}</b>',
    showarrow=False,
    xanchor='left',
    xshift=6,
    font=dict(color='#2E75B6', size=12, family='Arial')
)

fig.update_layout(
    title="China's CO2 emissions have surged past every other Asian nation since 2000",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    xaxis=dict(showgrid=False, title=''),
    yaxis=dict(gridcolor='#EEEEEE', gridwidth=1, title='CO2 Emissions (Mt)'),
    margin=dict(l=60, r=100, t=60, b=40)
)

fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [4]:
# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../data/co2_emissions.csv')

# Aggregate: average CO2 per region per year, then keep only 2000 and 2022
regional = (
    df.groupby(['Region', 'Year'])['CO2_Mt']
    .mean()
    .reset_index()
)
slope_df = regional[regional['Year'].isin([2000, 2022])]

# Determine direction of change per region
pivot = slope_df.pivot(index='Region', columns='Year', values='CO2_Mt')
pivot.columns.name = None
pivot = pivot.rename(columns={2000: 'val_2000', 2022: 'val_2022'})
pivot['increased'] = pivot['val_2022'] > pivot['val_2000']

# Colors: increased = red, decreased = steelblue
COLOR_UP   = '#C0392B'
COLOR_DOWN = '#2E75B6'

fig = go.Figure()

for region, row in pivot.iterrows():
    color = COLOR_UP if row['increased'] else COLOR_DOWN

    fig.add_trace(go.Scatter(
        x=['2000', '2022'],
        y=[row['val_2000'], row['val_2022']],
        mode='lines+markers+text',
        name=region,
        line=dict(color=color, width=2),
        marker=dict(size=7, color=color),
        text=[
            f'{row["val_2000"]:.0f}',                   # left label: value only
            f'{region}  {row["val_2022"]:.0f}'           # right label: name + value
        ],
        textposition=['middle left', 'middle right'],
        textfont=dict(size=10, color=color, family='Arial'),
        showlegend=False
    ))

fig.update_layout(
    title='Asia & Middle East surged; Europe & N. America cut emissions 2000–2022',
    xaxis=dict(
        tickvals=['2000', '2022'],
        ticktext=['2000', '2022'],
        showgrid=False,
        range=[-0.5, 2.2]          # extra room for right-side labels
    ),
    yaxis=dict(
        showgrid=False,
        showticklabels=False,      # endpoint labels make y-axis redundant
        title=''
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=80, r=180, t=60, b=40),
    height=600,
    width=900
)

fig.show()